# AeroDelay Regression Benchmark

The goal is not to build a production model; it is to verify whether the Gold-layer preprocessing and feature engineering contain useful signal for predicting departure delay minutes.

Main rules:
- Train only on departure rows.
- Target is `Departure_Delay_Reg_Target`.
- Split by time, not random split.
- Exclude leakage, identity, and raw label columns from predictors.


In [1]:
import warnings
import joblib
import os
from pathlib import Path

# Keep sklearn/joblib single-threaded for Windows sandbox stability.
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
import lightgbm as lgb
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 120)

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next((p for p in candidates if (p / "18_24520617_24520636_Data").exists()), Path.cwd().resolve())

GOLD = PROJECT_ROOT / "18_24520617_24520636_Data" /"Gold_layer"
DEPARTURE_DIR = GOLD / "Features" / "master_departure_features_gold_annotated.csv"
REPORT_DIR = GOLD / "Audit" / "model_training"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "Departure_Delay_Reg_Target"
TEST_START = pd.Timestamp("2026-03-01")
PASSENGER_ONLY = True
RANDOM_STATE = 108

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEPARTURE_DIR:", DEPARTURE_DIR)
print("REPORT_DIR:", REPORT_DIR)


PROJECT_ROOT: /Users/nguyenhung/PycharmProjects/DS108
DEPARTURE_DIR: /Users/nguyenhung/PycharmProjects/DS108/Data/Gold_layer/Features/master_departure_features_gold_annotated.csv
REPORT_DIR: /Users/nguyenhung/PycharmProjects/DS108/Data/Gold_layer/Audit/model_training


## 1. Load Gold Departure Data

Use `Data crawl/Gold_layer/Departure/*_flights_departure_gold_layer.csv` instead of the master file so the origin airport can be recovered from the file name. Arrival rows are intentionally excluded for the departure-delay regression target.


In [2]:
df_raw = pd.read_csv(DEPARTURE_DIR)
df_raw["Scheduled_Time"] = pd.to_datetime(df_raw["Scheduled_Time"], errors="coerce")

# Safe calendar features derived from scheduled time only.
df_raw["Scheduled_Hour"] = df_raw["Scheduled_Time"].dt.hour
df_raw["Scheduled_DayOfWeek"] = df_raw["Scheduled_Time"].dt.dayofweek
df_raw["Scheduled_Month"] = df_raw["Scheduled_Time"].dt.month
df_raw["Is_Weekend"] = df_raw["Scheduled_DayOfWeek"].isin([5, 6]).astype("Int64")

mask = (
    df_raw["Record_Type"].eq("Departure")
    & df_raw[TARGET].notna()
    & df_raw["Scheduled_Time"].notna()
)
if "Exclude_From_Propagation_Training" in df_raw.columns:
    mask &= ~df_raw["Exclude_From_Propagation_Training"].fillna(False).astype(bool)
if PASSENGER_ONLY and "Category" in df_raw.columns:
    mask &= df_raw["Category"].astype("string").str.lower().eq("passenger")

df = df_raw.loc[mask].copy()
df = df.sort_values("Scheduled_Time").reset_index(drop=True)

profile = pd.DataFrame([{
    "raw_rows": len(df_raw),
    "training_rows": len(df),
    "passenger_only": PASSENGER_ONLY,
    "scheduled_min": df["Scheduled_Time"].min(),
    "scheduled_max": df["Scheduled_Time"].max(),
    "target_mean": df[TARGET].mean(),
    "target_median": df[TARGET].median(),
    "target_p95": df[TARGET].quantile(0.95),
    "target_max": df[TARGET].max(),
}])
profile.to_csv(REPORT_DIR / "01_training_dataset_profile.csv", index=False)

display(profile)
display(df["Scheduled_Time"].dt.to_period("M").value_counts().sort_index().rename("rows_by_month").reset_index())
display(df[["Origin", TARGET]].groupby("Origin").agg(rows=(TARGET, "size"), median_delay=(TARGET, "median"), p95_delay=(TARGET, lambda s: s.quantile(0.95))).reset_index())


,raw_rows,training_rows,passenger_only,scheduled_min,scheduled_max,target_mean,target_median,target_p95,target_max
0,77758,72375,True,2025-12-15 18:00:00,2026-03-16 23:40:00,34.154515,23.0,113.0,720.0


,Scheduled_Time,rows_by_month
0,2025-12,12001
1,2026-01,22936
2,2026-02,24990
3,2026-03,12448


,Origin,rows,median_delay,p95_delay
0,DAD,11527,0.0,70.0
1,HAN,25600,21.0,97.0
2,SGN,35248,29.0,130.0


## 2. Define Leakage-Safe Feature Sets

Ablation groups follow `model_planning.md`:
- `F0_base`: schedule, airport, categorical, and load basics.
- `F1_turnaround`: adds model-safe turnaround features.
- `F2_lag`: adds propagation lag features.
- `F3_weather`: adds weather features.


In [3]:
LEAKAGE_OR_LABEL_COLUMNS = {
    "Departure_Delay", TARGET, "Actual_Time", "Flight_No", "Airline", "Scheduled_Tail",
    "Matched_Actual_Tail", "Swap_Match_Gap_Minutes", "Runway_Swap_Event",
    "Crawl_Date", "Status", "Checkin_Time", "Checkin_Counter", "Gate",
    "LLM_Delay_Code", "LLM_Delay_Reason"
}

F0_BASE = [
    "Airport", "IATA", "Airline_Type", "Aircraft_Type", "Category", "Time_of_Day",
    "Scheduled_Hour", "Scheduled_DayOfWeek", "Scheduled_Month", "Is_Weekend",
    "Peak_Hour_Indicator", "Is_Special_Days", "Is_Wide_Body", "Is_First_Flight",
    "Tail_Sequence_Day", "Standard_Turnaround",
    "Airport_Load_Factor", "Number_of_Flights_in_Last_Hour", "Is_Airport_Congested",
    "Is_Parallel_Usage", "Ground_Handling_Pressure", "Taxi_Out_Congestion",
    "A_CDM_TOBT_Deficit", "Destination_Congestion_Risk", "Previous_Station_Disruption"
]

F1_TURNAROUND = [
    "Turnaround_Buffer_Model", "Tail_Stagnation_Duration_Model",
    "Turnaround_Deficit_Min", "Is_Long_Ground_Turnaround"
]

F2_LAG = [
    "Prev_Departure_Delay_Tail_1", "Prev_Departure_Delay_Tail_2",
    "Rolling_Departure_Delay_Tail_3",
    "Prev_Turnaround_Buffer_Tail_1", "Prev_Turnaround_Buffer_Tail_2",
    "Rolling_Turnaround_Buffer_Tail_3",
    "Prev_Departure_Delay_Airport_1", "Rolling_Departure_Delay_Airport_3",
    "Airline_Avg_Delay_Rate", "Airline_Delay_Volatility"
]

WEATHER_FEATURES = [
    "Visibility_Severity_Score",
    "Is_Rain", "Is_Heavy_Rain","Is_Fog", "Is_Crosswind_10kt", "Is_Low_Ceiling_Risk",
    "Is_Thunderstorm_Risk", "Convective_Severity_Score", "Runway_Wet_Risk", "Forced_Runway_Swap_Risk", "Weather_Delay_Risk_Score", "Aviation_Operational_Risk_Score"
]

FEATURE_SETS = {
    "F0_base": F0_BASE,
    "F1_turnaround": F0_BASE + F1_TURNAROUND,
    "F2_lag": F0_BASE + F1_TURNAROUND + F2_LAG,
    "F3_weather": F0_BASE + F1_TURNAROUND + F2_LAG + WEATHER_FEATURES,
}

def available_features(columns, candidates):
    return [c for c in candidates if c in columns and c not in LEAKAGE_OR_LABEL_COLUMNS]

feature_audit_rows = []
for set_name, candidates in FEATURE_SETS.items():
    used = available_features(df.columns, candidates)
    missing = sorted(set(candidates) - set(used))
    leaked = sorted(set(used) & LEAKAGE_OR_LABEL_COLUMNS)
    feature_audit_rows.append({
        "feature_set": set_name,
        "candidate_count": len(candidates),
        "used_count": len(used),
        "missing_count": len(missing),
        "missing_features": ", ".join(missing),
        "leakage_overlap": ", ".join(leaked),
    })

feature_audit = pd.DataFrame(feature_audit_rows)
feature_audit.to_csv(REPORT_DIR / "02_feature_set_audit.csv", index=False)
display(feature_audit)


,feature_set,candidate_count,used_count,missing_count,missing_features,leakage_overlap
0,F0_base,25,23,2,"Airport, Is_Special_Days",
1,F1_turnaround,29,26,3,"Airport, Is_Special_Days, Turnaround_Deficit_Min",
2,F2_lag,39,36,3,"Airport, Is_Special_Days, Turnaround_Deficit_Min",
3,F3_weather,51,46,5,"Airport, Is_Crosswind_10kt, Is_Heavy_Rain, Is_...",


## 3. Time-Based Split

The benchmark uses a chronological split. By default, flights before March 2026 are train data and March 2026 is test data.


In [4]:
train_df = df[df["Scheduled_Time"] < TEST_START].copy()
test_df = df[df["Scheduled_Time"] >= TEST_START].copy()

if train_df.empty or test_df.empty:
    raise ValueError("Time split produced an empty train or test set. Check TEST_START and Scheduled_Time coverage.")

split_profile = pd.DataFrame([
    {"split": "train", "rows": len(train_df), "start": train_df["Scheduled_Time"].min(), "end": train_df["Scheduled_Time"].max(), "target_median": train_df[TARGET].median(), "target_p95": train_df[TARGET].quantile(0.95)},
    {"split": "test", "rows": len(test_df), "start": test_df["Scheduled_Time"].min(), "end": test_df["Scheduled_Time"].max(), "target_median": test_df[TARGET].median(), "target_p95": test_df[TARGET].quantile(0.95)},
])
split_profile.to_csv(REPORT_DIR / "03_time_split_profile.csv", index=False)
display(split_profile)


,split,rows,start,end,target_median,target_p95
0,train,59927,2025-12-15 18:00:00,2026-02-28 23:55:00,24.0,119.0
1,test,12448,2026-03-01 00:01:00,2026-03-16 23:40:00,19.0,73.0


## 4. Model Pipelines

All models use the same leakage-safe feature matrix. Numeric columns are median-imputed; categorical columns are imputed with `missing` and one-hot encoded. `Ridge` additionally scales numeric features.


In [5]:
def make_preprocessor(X: pd.DataFrame, scale_numeric: bool = False) -> ColumnTransformer:
    numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_pipe = Pipeline(numeric_steps)
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )


def make_models(X_sample: pd.DataFrame):
    return {
        "DummyMedian": Pipeline([
            ("preprocess", make_preprocessor(X_sample, scale_numeric=False)),
            ("model", DummyRegressor(strategy="median")),
        ]),
        "Ridge": Pipeline([
            ("preprocess", make_preprocessor(X_sample, scale_numeric=True)),
            ("model", Ridge(alpha=10.0, random_state=RANDOM_STATE)),
        ]),
        "ExtraTrees": Pipeline([
            ("preprocess", make_preprocessor(X_sample, scale_numeric=False)),
            ("model", ExtraTreesRegressor(
                n_estimators=120,
                min_samples_leaf=10,
                max_depth=15,
                max_features="sqrt",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ]),
        "HistGradientBoosting": Pipeline([
            ("preprocess", make_preprocessor(X_sample, scale_numeric=False)),
            ("model", HistGradientBoostingRegressor(
                max_iter=160,
                learning_rate=0.05,
                l2_regularization=0.05,
                max_leaf_nodes=31,
                random_state=RANDOM_STATE,
            )),
        ]),
        "Quantile_Regressor": Pipeline([
            ("preprocess", make_preprocessor(X_sample, scale_numeric=False)),
            ("model", HistGradientBoostingRegressor(
                loss="quantile",
                quantile=0.5,
                max_iter=150,
                max_leaf_nodes=31,
                random_state=42
            ))
        ]),
        "LightGBM": Pipeline([
            ("preprocess", make_preprocessor(X_sample, scale_numeric=False)),
            ("model", lgb.LGBMRegressor(
                objective="mae",
                n_estimators=150,
                random_state=42,
                n_jobs=-1
            ))
        ]),
        "Quantile_GBM_q0.5": Pipeline([
            ("preprocess", make_preprocessor(X_sample, scale_numeric=False)),
            ("model", HistGradientBoostingRegressor(
                loss="quantile", quantile=0.5, max_iter=150, max_leaf_nodes=31, random_state=42
            ))
        ]),
        "Huber_Quantile_LGBM": Pipeline([
            ("preprocess", make_preprocessor(X_sample, scale_numeric=False)),
            ("model", lgb.LGBMRegressor(
                objective="quantile",
                alpha=0.5,
                metric="mae",
                n_estimators=150,
                random_state=42,
                n_jobs=-1
            ))
        ])
    }


def evaluate_predictions(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MedianAE": median_absolute_error(y_true, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": r2_score(y_true, y_pred),
    }


## 5. Run Ablation Benchmark

This trains each model on each feature set and evaluates on the March 2026 test split. `MAE` is the primary metric because it is directly interpretable in minutes.


In [6]:
results = []
fitted_models = {}

for feature_set_name, candidates in FEATURE_SETS.items():
    features = available_features(df.columns, candidates)
    X_train = train_df[features].copy()
    y_train = train_df[TARGET].astype(float)
    X_test = test_df[features].copy()
    y_test = test_df[TARGET].astype(float)

    models = make_models(X_train)
    for model_name, pipeline in models.items():
        print(f"Training {feature_set_name} / {model_name} with {len(features)} features...")
        pipeline.fit(X_train, y_train)
        pred = np.clip(pipeline.predict(X_test), -60, 720)
        metrics = evaluate_predictions(y_test, pred)
        row = {
            "feature_set": feature_set_name,
            "model": model_name,
            "n_features": len(features),
            "train_rows": len(X_train),
            "test_rows": len(X_test),
            **metrics,
        }
        results.append(row)
        fitted_models[(feature_set_name, model_name)] = pipeline

results_df = pd.DataFrame(results).sort_values(["MAE", "RMSE"]).reset_index(drop=True)
results_df.to_csv(REPORT_DIR / "04_regression_benchmark_results.csv", index=False)
display(results_df)


Training F0_base / DummyMedian with 23 features...
Training F0_base / Ridge with 23 features...
Training F0_base / ExtraTrees with 23 features...
Training F0_base / HistGradientBoosting with 23 features...
Training F0_base / Quantile_Regressor with 23 features...
Training F0_base / LightGBM with 23 features...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005712 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 566
[LightGBM] [Info] Number of data points in the train set: 59927, number of used features: 148
[LightGBM] [Info] Start training from score 24.000000
Training F0_base / Quantile_GBM_q0.5 with 23 features...
Training F0_base / Huber_Quantile_LGBM with 23 features...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007010 seconds.
You can set `force_row_wise=true` to remove the overhead.
And

,feature_set,model,n_features,train_rows,test_rows,MAE,MedianAE,RMSE,R2
0,F3_weather,LightGBM,46,59927,12448,13.382671,7.458325,28.411542,0.240945
1,F2_lag,LightGBM,36,59927,12448,13.394784,7.442670,28.662329,0.227485
2,F3_weather,Huber_Quantile_LGBM,46,59927,12448,13.435217,7.549077,28.475164,0.237542
3,F2_lag,Huber_Quantile_LGBM,36,59927,12448,13.455813,7.544445,28.582252,0.231796
4,F3_weather,Quantile_Regressor,46,59927,12448,13.553964,7.542958,29.086211,0.204467
5,F3_weather,Quantile_GBM_q0.5,46,59927,12448,13.553964,7.542958,29.086211,0.204467
6,F2_lag,Quantile_Regressor,36,59927,12448,13.625273,7.606235,29.177481,0.199467
7,F2_lag,Quantile_GBM_q0.5,36,59927,12448,13.625273,7.606235,29.177481,0.199467
8,F1_turnaround,Huber_Quantile_LGBM,26,59927,12448,14.951762,8.745244,29.849184,0.162184
9,F1_turnaround,Quantile_Regressor,26,59927,12448,15.029745,8.943887,30.041403,0.151359


## 6. Segment Diagnostics

Evaluate the best test model by airport and month. This helps show whether the model is learning a global pattern only, or whether performance differs across SGN/HAN/DAD and time.


In [7]:
best = results_df.iloc[0]
best_key = (best["feature_set"], best["model"])
best_features = available_features(df.columns, FEATURE_SETS[best["feature_set"]])
best_model = fitted_models[best_key]

test_pred = np.clip(best_model.predict(test_df[best_features]), -60, 720)
predictions = test_df[["Origin", "Scheduled_Time", "IATA", "Aircraft_Type", "Category", TARGET]].copy()
predictions["prediction"] = test_pred
predictions["abs_error"] = (predictions[TARGET] - predictions["prediction"]).abs()
predictions["scheduled_month"] = predictions["Scheduled_Time"].dt.to_period("M").astype(str)

segment_rows = []
for group_cols in [["Origin"], ["scheduled_month"], ["Origin", "scheduled_month"]]:
    for keys, g in predictions.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {col: key for col, key in zip(group_cols, keys)}
        row.update({
            "grouping": "+".join(group_cols),
            "rows": len(g),
            "MAE": mean_absolute_error(g[TARGET], g["prediction"]),
            "MedianAE": median_absolute_error(g[TARGET], g["prediction"]),
            "RMSE": float(np.sqrt(mean_squared_error(g[TARGET], g["prediction"]))),
            "target_median": g[TARGET].median(),
            "prediction_median": g["prediction"].median(),
        })
        segment_rows.append(row)

segment_metrics = pd.DataFrame(segment_rows)
segment_metrics.to_csv(REPORT_DIR / "05_best_model_segment_metrics.csv", index=False)
predictions.to_csv(REPORT_DIR / "06_best_model_test_predictions.csv", index=False)

print("Best model:", best_key)
display(segment_metrics.sort_values(["grouping", "MAE"]))
display(predictions.sort_values("abs_error", ascending=False).head(20))


Best model: ('F3_weather', 'LightGBM')


,Origin,grouping,rows,MAE,MedianAE,RMSE,target_median,prediction_median,scheduled_month
0,DAD,Origin,2032,8.930922,3.134345,22.852837,0.0,3.554300,NaN
1,HAN,Origin,4478,13.682604,7.735148,29.895177,19.0,19.219426,NaN
2,SGN,Origin,5938,14.679885,8.772002,28.974036,23.0,25.218858,NaN
4,DAD,Origin+scheduled_month,2032,8.930922,3.134345,22.852837,0.0,3.554300,2026-03
5,HAN,Origin+scheduled_month,4478,13.682604,7.735148,29.895177,19.0,19.219426,2026-03
6,SGN,Origin+scheduled_month,5938,14.679885,8.772002,28.974036,23.0,25.218858,2026-03
3,NaN,scheduled_month,12448,13.382671,7.458325,28.411542,19.0,20.619388,2026-03


,Origin,Scheduled_Time,IATA,Aircraft_Type,Category,Departure_Delay_Reg_Target,prediction,abs_error,scheduled_month
69131,HAN,2026-03-12 16:20:00,DAD,A321,passenger,634.0,13.097669,620.902331,2026-03
68170,SGN,2026-03-11 11:15:00,HKT,A321,passenger,633.0,40.961690,592.038310,2026-03
68362,HAN,2026-03-11 15:55:00,CXR,A321,passenger,480.0,31.169895,448.830105,2026-03
68997,HAN,2026-03-12 12:55:00,CXR,A321,passenger,536.0,90.759643,445.240357,2026-03
71714,SGN,2026-03-16 04:35:00,BKK,A321,passenger,442.0,12.191426,429.808574,2026-03
72092,SGN,2026-03-16 14:50:00,MNL,A20N,passenger,429.0,26.725999,402.274001,2026-03
60855,DAD,2026-03-02 03:00:00,PUS,A333,passenger,375.0,3.156419,371.843581,2026-03
68879,SGN,2026-03-12 10:05:00,DAD,A321,passenger,430.0,58.481560,371.518440,2026-03
63355,HAN,2026-03-05 08:00:00,PQC,B789,passenger,382.0,25.908465,356.091535,2026-03
71201,SGN,2026-03-15 11:30:00,CXR,AT75,passenger,381.0,29.919457,351.080543,2026-03


In [8]:
# 1. Khởi tạo thư mục lưu trữ
model_dir = PROJECT_ROOT / "18_24520617_24520636_Source code" / "3-Feature Engineering & Model" / "models"
model_dir.mkdir(parents=True, exist_ok=True)

# 2. KIỂM TRA BẢNG KẾT QUẢ BENCHMARK
if 'results_df' in locals() or 'results_df' in globals():
    # Sắp xếp kết quả để tìm ra mô hình có MAE thấp nhất (Tốt nhất)
    # Loại bỏ mô hình 'DummyMedian' vì đây chỉ là mô hình baseline để so sánh
    valid_results = results_df[results_df['model'] != 'DummyMedian']

    if not valid_results.empty:
        # Lấy dòng đầu tiên có MAE nhỏ nhất
        best_row = valid_results.sort_values(by='MAE', ascending=True).iloc[0]

        # Tự động trích xuất tên tập feature và tên thuật toán tốt nhất
        best_feature_set = best_row['feature_set']
        best_model_name = best_row['model']
        best_mae = best_row['MAE']
        best_r2 = best_row['R2']

        print("=" * 60)
        print("🤖 [HỆ THỐNG PHÁT HIỆN MÔ HÌNH TỐT NHẤT TỰ ĐỘNG]")
        print(f"   - Tập Đặc Trưng Tốt Nhất: {best_feature_set}")
        print(f"   - Thuật Toán Tốt Nhất:   {best_model_name}")
        print(f"   - Chỉ số MAE đạt được:   {best_mae:.4f}")
        print(f"   - Chỉ số R2 đạt được:    {best_r2:.4f}")
        print("=" * 60)

        # 3. Lấy đối tượng mô hình thực tế từ từ điển fitted_models
        if (best_feature_set, best_model_name) in fitted_models:
            best_model_obj = fitted_models[(best_feature_set, best_model_name)]

            # Xuất file Trọng số Mô hình (.pkl)
            model_path = model_dir / "best_aerodelay_model.pkl"
            joblib.dump(best_model_obj, model_path)
            print(f"[V] Đã đóng gói mô hình tốt nhất tại: {model_path}")

            # --- PHẦN BỔ SUNG LƯU FEATURE LIST ---
            # 4. Trích xuất và xuất danh sách các feature thực tế đã dùng
            best_candidates = FEATURE_SETS[best_feature_set]
            best_features_list = available_features(df.columns, best_candidates)

            feature_path = model_dir / "best_model_features.pkl"
            joblib.dump(best_features_list, feature_path)
            print(f"[V] Đã lưu danh sách {len(best_features_list)} features tại: {feature_path}")
            # -------------------------------------

        else:
            print(f"[!] Lỗi: Không tìm thấy cặp ({best_feature_set}, {best_model_name}) trong fitted_models.")
    else:
        print("[!] Lỗi: Bảng kết quả rỗng hoặc chỉ chứa mô hình Dummy.")
else:
    print("[!] Lỗi: Không tìm thấy DataFrame 'results_df'.")

🤖 [HỆ THỐNG PHÁT HIỆN MÔ HÌNH TỐT NHẤT TỰ ĐỘNG]
   - Tập Đặc Trưng Tốt Nhất: F3_weather
   - Thuật Toán Tốt Nhất:   LightGBM
   - Chỉ số MAE đạt được:   13.3827
   - Chỉ số R2 đạt được:    0.2409
[V] Đã đóng gói mô hình tốt nhất tại: /Users/nguyenhung/PycharmProjects/DS108/Source code/3-Feature Engineering & Model/models/best_aerodelay_model.pkl
[V] Đã lưu danh sách 46 features tại: /Users/nguyenhung/PycharmProjects/DS108/Source code/3-Feature Engineering & Model/models/best_model_features.pkl


## 7. Interpretation Checklist

Use the generated CSV files in `Data crawl/Gold_layer/Audit/model_training/` when writing the benchmark section:

- `01_training_dataset_profile.csv`: confirms row counts and target distribution.
- `02_feature_set_audit.csv`: confirms which planned features exist in Gold.
- `03_time_split_profile.csv`: documents the chronological split.
- `04_regression_benchmark_results.csv`: main ablation/model comparison.
- `05_best_model_segment_metrics.csv`: airport/month diagnostics.
- `06_best_model_test_predictions.csv`: row-level predictions for error inspection.

The strongest valid result is not necessarily the lowest global MAE alone. Prefer a model that improves over `DummyMedian`, behaves consistently across airports, and does not rely on leakage-risk or raw identity columns.
